# Notebook to load JSON and generate .md

In [1]:
from pathlib import Path
import json

In [4]:
ROOT = Path.cwd()

RELEASE_DIR = ROOT.parent.parent / "cloud-releases"
OUT_FILE = ROOT / "docs" / "rosetta-stone" / "releases.md"

print("Release folder:", RELEASE_DIR)
print("Output file:", OUT_FILE)
print("Release folder exists:", RELEASE_DIR.exists())

Release folder: /Users/amaraalexander/Documents/GitHub/cloud-releases
Output file: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab/scripts/docs/rosetta-stone/releases.md
Release folder exists: True


In [9]:
releases = []

for file in RELEASE_DIR.glob("*.json"):
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # If JSON is shaped like:
    # {
    #   "v1.0.0": {...},
    #   "v1.1.0": {...}
    # }
    if isinstance(data, dict):
        for release_id, release_info in data.items():
            if isinstance(release_info, dict):
                release_info["id"] = release_id
                releases.append(release_info)


print(f"Loaded {len(releases)} releases")
releases[:2]

Loaded 13 releases


[{'release_version': 'v1.0.0',
  'cde_version': 'v2.1',
  'release_doi': '10.5281/zenodo.11585274',
  'datasets': [{'name': 'hafler-pmdbs-sn-rnaseq-pfc',
    'doi': '10.5281/zenodo.15490150',
    'dataset_version': 'v1.0'},
   {'name': 'lee-pmdbs-sn-rnaseq',
    'doi': '10.5281/zenodo.16744323',
    'dataset_version': 'v1.0'},
   {'name': 'jakobsson-pmdbs-sn-rnaseq',
    'doi': '10.5281/zenodo.15162834',
    'dataset_version': 'v1.0'},
   {'name': 'scherzer-pmdbs-sn-rnaseq-mtg',
    'doi': '10.5281/zenodo.16885831',
    'dataset_version': 'v1.0'},
   {'name': 'cohort-pmdbs-sc-rnaseq',
    'doi': '10.5281/zenodo.14373047',
    'dataset_version': 'v1.0.0'}],
  'collections': {'pmdbs-sc-rnaseq': {'name': 'pmdbs-sc-rnaseq',
    'doi': '10.5281/zenodo.14373047',
    'version': 'v1.0.0'}},
  'created': '2026-03-24T13:41:44.125113',
  'metadata': {'total_datasets': 5,
   'total_collections': 1,
   'source': 'v1.0.0/datasets.csv'},
  'new_datasets': [{'name': 'hafler-pmdbs-sn-rnaseq-pfc',
    

In [33]:
from pathlib import Path
import json
import re
import html


# ----------------------------
# Find Learning Lab repo root
# ----------------------------

def find_repo_root(start_path):
    start_path = Path(start_path).resolve()

    for path in [start_path] + list(start_path.parents):
        if (path / "mkdocs.yml").exists():
            return path

    raise FileNotFoundError("Could not find mkdocs.yml. Run this from inside the Learning Lab repo.")


ROOT = find_repo_root(Path.cwd())

RELEASE_REPO = ROOT.parent / "cloud-releases"
RELEASE_DIR = RELEASE_REPO / "records"

if not RELEASE_DIR.exists():
    RELEASE_DIR = RELEASE_REPO

OUT_FILE = ROOT / "docs" / "rosetta-stone" / "releases.md"
JS_FILE = ROOT / "docs" / "javascripts" / "release-filter.js"

print("Learning Lab root:", ROOT)
print("Release JSON folder:", RELEASE_DIR)
print("Output Markdown:", OUT_FILE)
print("Output JavaScript:", JS_FILE)


# ----------------------------
# Helpers
# ----------------------------

def esc(value):
    if value is None:
        return ""
    return html.escape(str(value), quote=True)


def safe_id(value):
    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9]+", "-", value)
    return value.strip("-") or "release"


def doi_url(doi):
    if not doi:
        return ""

    doi = str(doi).strip()

    if doi.startswith("http://") or doi.startswith("https://"):
        return doi

    return f"https://doi.org/{doi}"


def doi_link(doi):
    if not doi:
        return "TBD"

    doi = str(doi).strip()
    url = doi_url(doi)
    label = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")

    return f'<a href="{esc(url)}" target="_blank" rel="noopener">{esc(label)}</a>'


def get_release_version(release):
    return (
        release.get("release_version")
        or release.get("version")
        or release.get("id")
        or "unknown-version"
    )


def get_release_id(release):
    return (
        release.get("id")
        or release.get("release_version")
        or release.get("version")
        or "unknown-release"
    )


def version_key(release):
    version = get_release_version(release)
    numbers = re.findall(r"\d+", str(version))

    if not numbers:
        return (0,)

    return tuple(int(number) for number in numbers)


def normalize_release_records(data, source_file):
    records = []

    # Shape:
    # {
    #   "v1.0.0": {...},
    #   "v2.0.0": {...}
    # }
    if isinstance(data, dict) and not any(
        key in data for key in ["release_version", "version", "datasets", "cde_version", "release_doi"]
    ):
        for key, value in data.items():
            if isinstance(value, dict):
                record = dict(value)
                record.setdefault("id", key)
                record.setdefault("release_version", key)
                record["_source_file"] = str(source_file)
                records.append(record)

    return records


# ----------------------------
# Load release JSON
# ----------------------------

releases = []

if not RELEASE_DIR.exists():
    raise FileNotFoundError(f"Could not find release folder: {RELEASE_DIR}")

for file in RELEASE_DIR.rglob("*.json"):
    try:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        releases.extend(normalize_release_records(data, file))

    except Exception as error:
        print(f"Skipping {file}: {error}")

if not releases:
    raise ValueError("No release records found.")

releases = sorted(releases, key=version_key, reverse=True)

print(f"Loaded {len(releases)} releases")
print("Newest:", get_release_version(releases[0]))


# ----------------------------
# Generate releases.md
# ----------------------------

lines = [
    "# CRN Cloud Releases",
    "",
    "Browse CRN Cloud release records generated from release JSON. Use the filter to search by release, CDE version, dataset name, dataset version, or DOI.",
    "",
    '<input id="releaseSearch" class="release-search" type="text" placeholder="Filter releases, datasets, versions, or DOIs...">',
    "",
    '<p id="releaseCount" class="release-count"></p>',
    "",
    "<style>",
    ".release-search {",
    "  width: 100%;",
    "  padding: 0.75rem;",
    "  margin: 1rem 0 0.5rem 0;",
    "  border: 1px solid var(--md-default-fg-color--lightest);",
    "  border-radius: 0.45rem;",
    "  font-size: 1rem;",
    "}",
    ".release-count {",
    "  margin: 0 0 1rem 0;",
    "  color: var(--md-default-fg-color--light);",
    "}",
    ".release-grid {",
    "  display: grid;",
    "  grid-template-columns: repeat(auto-fit, minmax(260px, 1fr));",
    "  gap: 1rem;",
    "}",
    ".release-card {",
    "  border: 1px solid var(--md-default-fg-color--lightest);",
    "  border-radius: 0.7rem;",
    "  padding: 1rem;",
    "  cursor: pointer;",
    "  background: var(--md-default-bg-color);",
    "}",
    ".release-card:hover {",
    "  border-color: var(--md-accent-fg-color);",
    "  box-shadow: 0 0.2rem 0.6rem rgba(0,0,0,0.08);",
    "}",
    ".release-card h3 {",
    "  margin-top: 0;",
    "  margin-bottom: 0.35rem;",
    "}",
    ".release-card p {",
    "  margin: 0.35rem 0;",
    "}",
    ".release-modal {",
    "  display: none;",
    "  position: fixed;",
    "  z-index: 9999;",
    "  left: 0;",
    "  top: 0;",
    "  width: 100%;",
    "  height: 100%;",
    "  background: rgba(0, 0, 0, 0.45);",
    "  align-items: flex-start;",
    "  justify-content: center;",
    "  overflow: auto;",
    "  padding: 3rem 1rem;",
    "}",
    ".release-modal-content {",
    "  width: min(950px, 95vw);",
    "  background: var(--md-default-bg-color);",
    "  border-radius: 0.75rem;",
    "  padding: 1.25rem;",
    "  box-shadow: 0 0.4rem 1.5rem rgba(0,0,0,0.25);",
    "}",
    ".release-modal-close {",
    "  float: right;",
    "  font-size: 1.6rem;",
    "  font-weight: bold;",
    "  cursor: pointer;",
    "}",
    ".release-table {",
    "  width: 100%;",
    "  border-collapse: collapse;",
    "  margin-top: 1rem;",
    "}",
    ".release-table th, .release-table td {",
    "  border-bottom: 1px solid var(--md-default-fg-color--lightest);",
    "  padding: 0.55rem;",
    "  text-align: left;",
    "  vertical-align: top;",
    "}",
    "</style>",
    "",
    '<div class="release-grid" id="releaseGrid">',
]

for index, release in enumerate(releases):
    release_id = get_release_id(release)
    release_version = get_release_version(release)
    cde_version = release.get("cde_version", "")
    release_doi = release.get("release_doi", "")
    datasets = release.get("datasets", [])

    if not isinstance(datasets, list):
        datasets = []

    modal_id = f"release-modal-{safe_id(release_id)}-{index}"

    dataset_search = " ".join(
        [
            f"{dataset.get('name', '')} {dataset.get('dataset_version', '')} {dataset.get('doi', '')}"
            for dataset in datasets
            if isinstance(dataset, dict)
        ]
    )

    search_text = " ".join(
        [
            str(release_id),
            str(release_version),
            str(cde_version),
            str(release_doi),
            dataset_search,
        ]
    ).lower()

    lines.extend([
        f'<div class="release-card" data-modal="{esc(modal_id)}" data-search="{esc(search_text)}">',
        f"  <h3>CRN Cloud Release {esc(release_version)}</h3>",
        f"  <p><code>{esc(release_id)}</code></p>",
        f"  <p><strong>Release version:</strong> {esc(release_version)}</p>",
        f"  <p><strong>CDE version:</strong> {esc(cde_version) if cde_version else 'TBD'}</p>",
        f"  <p><strong>Release DOI:</strong> {doi_link(release_doi)}</p>",
        f"  <p><em>Click card to view {len(datasets)} dataset(s)</em></p>",
        "</div>",
    ])

lines.append("</div>")
lines.append("")

for index, release in enumerate(releases):
    release_id = get_release_id(release)
    release_version = get_release_version(release)
    cde_version = release.get("cde_version", "")
    release_doi = release.get("release_doi", "")
    datasets = release.get("datasets", [])

    if not isinstance(datasets, list):
        datasets = []

    modal_id = f"release-modal-{safe_id(release_id)}-{index}"

    lines.extend([
        f'<div id="{esc(modal_id)}" class="release-modal">',
        '  <div class="release-modal-content">',
        '    <span class="release-modal-close" data-close="true">&times;</span>',
        f"    <h2>CRN Cloud Release {esc(release_version)}</h2>",
        f"    <p><strong>Release ID:</strong> <code>{esc(release_id)}</code></p>",
        f"    <p><strong>CDE version:</strong> {esc(cde_version) if cde_version else 'TBD'}</p>",
        f"    <p><strong>Release DOI:</strong> {doi_link(release_doi)}</p>",
        "    <h3>Datasets in this release</h3>",
        '    <table class="release-table">',
        "      <thead>",
        "        <tr>",
        "          <th>Dataset</th>",
        "          <th>Dataset version</th>",
        "          <th>DOI</th>",
        "        </tr>",
        "      </thead>",
        "      <tbody>",
    ])

    if datasets:
        for dataset in datasets:
            if not isinstance(dataset, dict):
                continue

            lines.extend([
                "        <tr>",
                f"          <td><code>{esc(dataset.get('name', ''))}</code></td>",
                f"          <td>{esc(dataset.get('dataset_version', '')) or 'TBD'}</td>",
                f"          <td>{doi_link(dataset.get('doi', ''))}</td>",
                "        </tr>",
            ])
    else:
        lines.extend([
            "        <tr>",
            '          <td colspan="3">No datasets listed for this release.</td>',
            "        </tr>",
        ])

    lines.extend([
        "      </tbody>",
        "    </table>",
        "  </div>",
        "</div>",
    ])

markdown_text = "\n".join(lines)

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
OUT_FILE.write_text(markdown_text, encoding="utf-8")


# ----------------------------
# Generate JavaScript separately
# ----------------------------

js_text = """
function initializeReleasePage() {
  const searchInput = document.getElementById("releaseSearch");
  const releaseCount = document.getElementById("releaseCount");
  const cards = Array.from(document.querySelectorAll(".release-card"));
  const modals = Array.from(document.querySelectorAll(".release-modal"));

  if (!cards.length) {
    return;
  }

  function updateCount(visibleCount) {
    if (releaseCount) {
      releaseCount.textContent = visibleCount + " of " + cards.length + " releases shown";
    }
  }

  function closeAllModals() {
    modals.forEach(function (modal) {
      modal.style.display = "none";
    });
  }

  cards.forEach(function (card) {
    card.addEventListener("click", function (event) {
      if (event.target.closest("a")) {
        return;
      }

      const modalId = card.getAttribute("data-modal");
      const modal = document.getElementById(modalId);

      if (modal) {
        modal.style.display = "flex";
      }
    });
  });

  modals.forEach(function (modal) {
    modal.addEventListener("click", function (event) {
      if (event.target === modal || event.target.getAttribute("data-close") === "true") {
        closeAllModals();
      }
    });
  });

  document.addEventListener("keydown", function (event) {
    if (event.key === "Escape") {
      closeAllModals();
    }
  });

  if (searchInput) {
    searchInput.addEventListener("input", function () {
      const query = searchInput.value.toLowerCase().trim();
      let visibleCount = 0;

      cards.forEach(function (card) {
        const text = card.getAttribute("data-search") || "";
        const isVisible = text.includes(query);

        card.style.display = isVisible ? "block" : "none";

        if (isVisible) {
          visibleCount += 1;
        }
      });

      updateCount(visibleCount);
    });
  }

  updateCount(cards.length);
}

if (typeof document$ !== "undefined") {
  document$.subscribe(function () {
    initializeReleasePage();
  });
} else {
  document.addEventListener("DOMContentLoaded", initializeReleasePage);
}
"""

JS_FILE.parent.mkdir(parents=True, exist_ok=True)
JS_FILE.write_text(js_text.strip() + "\n", encoding="utf-8")

print(f"Wrote: {OUT_FILE}")
print(f"Wrote: {JS_FILE}")

Learning Lab root: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab
Release JSON folder: /Users/amaraalexander/Documents/GitHub/cloud-releases
Output Markdown: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab/docs/rosetta-stone/releases.md
Output JavaScript: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab/docs/javascripts/release-filter.js
Loaded 23 releases
Newest: v4.1.1
Wrote: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab/docs/rosetta-stone/releases.md
Wrote: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab/docs/javascripts/release-filter.js


In [31]:
OUT_FILE = ROOT.parent / "docs" / "rosetta-stone" / "releases.md"

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
OUT_FILE.write_text(markdown_text, encoding="utf-8")

print(f"Wrote: {OUT_FILE}")

Wrote: /Users/amaraalexander/Documents/GitHub/docs/rosetta-stone/releases.md
